# Analyse des images du jeu de données

In [2]:
import pandas as pd

df = pd.read_csv('image_list.csv', dtype={"file":str})

df.head()

,category,type,quality,file,extension,width,height,mode,bands,type_image,mask,mask_bands,mask_mode,mask_type
0,bottle,test,broken_large,000,.png,900,900,RGB,3,C,True,1.0,L,G
1,bottle,test,broken_large,001,.png,900,900,RGB,3,C,True,1.0,L,G
2,bottle,test,broken_large,002,.png,900,900,RGB,3,C,True,1.0,L,G
3,bottle,test,broken_large,003,.png,900,900,RGB,3,C,True,1.0,L,G
4,bottle,test,broken_large,004,.png,900,900,RGB,3,C,True,1.0,L,G


## Analyse des extensions

In [3]:
df["extension"].unique()

# Il n'y a que des png, donc la colonne est inutile

<StringArray>
['.png']
Length: 1, dtype: str

In [4]:
df = df.drop("extension", axis=1)

## Analyse des masques

In [5]:
df[["mask_type", "mask_bands", "mask_mode"]].nunique()
# Les masques sont tous en noir&blanc avec une seule bande. Donc on peut garder juste la colonne mask True/False

mask_type     1
mask_bands    1
mask_mode     1
dtype: int64

In [6]:
df = df.drop(["mask_type", "mask_bands", "mask_mode"],axis=1)

# Analyse des redondances entre mode / bands / type_image

In [7]:
df.groupby(["mode", "bands", "type_image"]).count()
# J'ai effectivement une redondance, le nombre de bandes suffit

,,,category,type,quality,file,width,height,mask
mode,bands,type_image,,,,,,,
L,1,G,1213,1213,1213,1213,1213,1213,1213
RGB,3,C,4141,4141,4141,4141,4141,4141,4141


In [8]:
df = df.drop(["mode", "type_image"],axis=1)

Je vérifie que j'ai un masque si et seulement si quality != "good"

In [9]:
(~(df["mask"]==(df['quality']!='good'))).sum()
# Cela confirme l'hypothèse, donc on peut supprimer la colonne mask

df = df.drop("mask", axis=1)

# Analyse des tailles et types d'images par catégories

In [10]:
df.groupby("category").agg({"width":("min", "max"), "height": ("min", "max"), "bands": ("min", "max")})

# On a une homogénéité dans les dimensions et types des images par catégorie
# De plus, on constate que toutes les images sont carrées

width       height       bands    
             min   max    min   max   min max
category                                     
bottle       900   900    900   900     3   3
cable       1024  1024   1024  1024     3   3
capsule     1000  1000   1000  1000     3   3
carpet      1024  1024   1024  1024     3   3
grid        1024  1024   1024  1024     1   1
hazelnut    1024  1024   1024  1024     3   3
leather     1024  1024   1024  1024     3   3
metal_nut    700   700    700   700     3   3
pill         800   800    800   800     3   3
screw       1024  1024   1024  1024     1   1
tile         840   840    840   840     3   3
toothbrush  1024  1024   1024  1024     3   3
transistor  1024  1024   1024  1024     3   3
wood        1024  1024   1024  1024     3   3
zipper      1024  1024   1024  1024     1   1

In [11]:
(df["width"]!=df["height"]).sum()
# On confirme qu'on a bien toujours la largeur = hauteur

np.int64(0)

In [12]:
df = df.drop(["width"],axis=1)
df = df.rename({"height":"dimension"}, axis=1)

In [13]:
df.head()

,category,type,quality,file,dimension,bands
0,bottle,test,broken_large,000,900,3
1,bottle,test,broken_large,001,900,3
2,bottle,test,broken_large,002,900,3
3,bottle,test,broken_large,003,900,3
4,bottle,test,broken_large,004,900,3


In [14]:
import plotly.graph_objects as go
import plotly.io as pio
import numpy as np

try:
    import nbformat  # noqa: F401
except ImportError:
    pio.renderers.default = "browser"

train = df[df["type"] == "train"].groupby("category").agg(count=("file", "count"))
test_good = df[(df["type"] == "test") & (df["quality"] == "good")].groupby("category").agg(count=("file", "count"))
test_bad = df[df["quality"] != "good"].groupby(["category", "quality"]).agg(count=("file", "count"))

bad_pivot = test_bad["count"].unstack("quality").fillna(0)
categories = train.index

x = np.arange(len(categories))
x_good = x-0.2
x_bad  = x+0.2

fig = go.Figure()

# Pile good/train
fig.add_bar(
    x=x_good,
    y=test_good["count"],
    name="test_good",
    marker=dict(color="green")
)

fig.add_bar(
    x=x_good,
    y=train["count"],
    name="train",
    marker=dict(color="lightgreen")
)

# Pile bad
for quality in bad_pivot.columns:
    fig.add_bar(
        x=x_bad,
        y=bad_pivot[quality],
        name=f"bad - {quality}",
    )

fig.update_xaxes(
    tickvals=x,
    ticktext=categories
)

fig.update_layout(
    barmode="stack",
    height=600,
    showlegend=False
)

fig.add_annotation(
    text="train (good)",
    xref="x", yref="y",
    x=4.8, y=350,
    axref="x", ayref="y",
    ax=4.5, ay=370,
    showarrow=True, 
    xanchor="right",
    font=dict(size=14, color="black")
)
fig.add_annotation(
    text="test (good)",
    xref="x", yref="y",
    x=4.8, y=25,
    axref="x", ayref="y",
    ax=4.5, ay=120,
    showarrow=True, 
    xanchor="right",
    font=dict(size=14, color="black")
)
fig.add_annotation(
    text="defects",
    xref="x", yref="y",
    x=5.2, y=68,
    axref="x", ayref="y",
    ax=5.3, ay=120,
    showarrow=True, 
    xanchor="center",
    font=dict(size=14, color="black")
)

fig.update_layout(
    title="Répartition des images par catégorie et par type (train/test) et qualité (good/bad)",
    xaxis_title="Catégorie",
    yaxis_title="Nombre d'images"
)

fig.show()

In [15]:
# Nombre d'erreurs par catégorie et par type d'erreur
counts=df[df["quality"]!="good"].groupby(["category", "quality"]).agg(count=('file','count')).reset_index()

fig=go.Figure()
fig.add_trace(go.Box(
    x=counts["category"],
    y=counts["count"]
))
fig.update_layout(
    title="Distribution du nombre d'images montrant des défauts, par quality",
    xaxis_title="Catégorie",
    yaxis_title="Nombre d'images montrant des défauts, par quality"
)
fig.show()

In [16]:
# Nombre d'erreurs par catégorie et par type d'erreur
counts_cat=df[df["quality"]!="good"].groupby(["category"]).agg(count=('file','count')).reset_index()

fig=go.Figure()
fig.add_trace(go.Box(
    x=counts_cat["count"],
    orientation="h"
))
fig.update_layout(
    title="Distribution du nombre d'images montrant des défauts, par catégorie",
    yaxis_title="Total",
    xaxis_title="Nombre d'images montrant des défauts, par catégorie"
)
fig.show()

In [17]:
# Nombre d'image "train" par catégorie
counts_cat=df[df["type"]=="train"].groupby(["category"]).agg(count=('file','count')).reset_index()

fig=go.Figure()
fig.add_trace(go.Box(
    x=counts_cat["count"], 
    orientation="h"
))
fig.update_layout(
    title="Distribution du nombre d'images d'entraînement par catégorie",
    yaxis_title="Total",
    xaxis_title="Nombre d'images 'train' par catégorie"
)
fig.show()